In [1]:
import mne
import matplotlib.pyplot as plt
from pathlib import Path

import pandas as pd

ROOT = Path().resolve().parent

set_file1 = ROOT / "sub-Rc01" / "ses-1" / "eeg" / "sub-Rc01_ses-1_task-SmartickDataset_run-1_eeg.set"
set_file2 = ROOT / "sub-Rc01" / "ses-1" / "eeg" / "sub-Rc01_ses-1_task-SmartickDataset_run-2_eeg.set"

raw1 = mne.io.read_raw_eeglab(set_file1, preload=True)
raw2 = mne.io.read_raw_eeglab(set_file2, preload=True)

data1 = raw1.get_data()
data2 = raw2.get_data()

print(raw1)
print("Channels:", len(raw1.ch_names))
print("Sampling frequency:", raw1.info["sfreq"])
print("Duration:", raw1.times[-1], "seconds")
print("Number of samples:", raw1.n_times)
print("Channel names:")
print(raw1.ch_names)

print(raw2)
print("Channels:", len(raw2.ch_names))
print("Sampling frequency:", raw2.info["sfreq"])
print("Duration:", raw2.times[-1], "seconds")
print("Number of samples:", raw2.n_times)
print("Channel names:")
print(raw2.ch_names)

FileNotFoundError: input_fname does not exist: "/Users/ananyosen/Documents/sub-Rc01/ses-1/eeg/sub-Rc01_ses-1_task-SmartickDataset_run-1_eeg.set"

In [ ]:
# EEG preprocessing

raw_clean1 = raw1.copy()

# Band-pass filtering
raw_clean1.filter(
    l_freq=0.5,
    h_freq=50.0,
    fir_design="firwin"
)

# Notch filter for power-line interference
raw_clean1.notch_filter(freqs=50)

raw_clean2 = raw2.copy()

# Band-pass filtering
raw_clean2.filter(
    l_freq=0.5,
    h_freq=50.0,
    fir_design="firwin"
)

# Notch filter for power-line interference
raw_clean2.notch_filter(freqs=50)



print("Preprocessing completed.")
print("High-pass filter : 0.5 Hz")
print("Low-pass filter  : 50 Hz")
print("Notch filter     : 50 Hz")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data1 = raw_clean1.get_data()
times1 = raw_clean1.times

data2 = raw_clean2.get_data()
times2 = raw_clean2.times

event_time = 131.2617188
end_time = event_time + 2

fig, axes = plt.subplots(1, 2, figsize=(18, 12))

spacing = 5

for i in range(data1.shape[0]):
    signal1 = data1[i]
    signal2 = data2[i]

    # Normalize each channel
    signal1 = (signal1 - np.mean(signal1)) / np.std(signal1)
    signal2 = (signal2 - np.mean(signal2)) / np.std(signal2)

    axes[0].plot(
        times1,
        signal1 + i * spacing,
        linewidth=0.5
    )

    axes[1].plot(
        times2,
        signal2 + i * spacing,
        linewidth=0.5
    )

axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Channels")
axes[0].set_title("Baseline Run", fontsize=14)

axes[0].set_yticks(
    np.arange(0, data1.shape[0] * spacing, spacing),
    raw_clean1.ch_names
)

axes[0].set_xlim(event_time, end_time)
axes[0].grid(alpha=0.2)

axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel(" ")
axes[1].set_title("Activity Run", fontsize=14)

axes[1].set_yticks(
    np.arange(0, data2.shape[0] * spacing, spacing),
    raw_clean2.ch_names
)

axes[1].set_xlim(event_time, end_time)
axes[1].grid(alpha=0.2)

fig.suptitle("EEG Signals from Two Different Runs", fontsize=16)

plt.tight_layout()
plt.show()

In [ ]:
selected_channels = ["T8"]

raw_clean1.compute_psd(
    fmin=0.5,
    fmax=50,
    n_fft=2048
).plot(
    picks=selected_channels,
    average=False
)

raw_clean2.compute_psd(
    fmin=0.5,
    fmax=50,
    n_fft=2048
).plot(
    picks=selected_channels,
    average=False
)

plt.show()

In [ ]:
channels_to_plot = ["Fz", "Cz", "Pz", "Oz"]

fig, axes = plt.subplots(
    len(channels_to_plot),
    1,
    figsize=(16, 10),
    sharex=True
)

for ax, ch in zip(axes, channels_to_plot):

    idx = raw2.ch_names.index(ch)

    ax.plot(
        raw2.times,
        data2[idx] * 1e6,
        linewidth=0.5
    )

    ax.axvline(
        raw2.times[-1]/2,
        linestyle="--",
        linewidth=1
    )

    ax.set_ylabel(f"{ch}\n(µV)")
    ax.grid(alpha=0.2)

axes[-1].set_xlabel("Time (seconds)")

fig.suptitle(
    "EEG Signal for Selected Channels",
    fontsize=14
)

plt.tight_layout()
plt.show()